In [ ]:
!python -m playwright install chromium

In [1]:
import asyncio
from playwright.async_api import async_playwright

async def save_state():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        await page.goto("https://www.threads.com/")
        print("1) Log in in the opened browser window.")
        print("2) When you are fully logged in, come back here and press Enter.")
        input()

        await context.storage_state(path="threads_state.json")
        await browser.close()
        print("Saved login state to: threads_state.json")

await save_state()


1) Log in in the opened browser window.
2) When you are fully logged in, come back here and press Enter.
Saved login state to: threads_state.json


In [3]:
import asyncio, json
from playwright.async_api import async_playwright

In [4]:

TARGET_USERNAME = "nike"          # <-- change this
MAX_SCROLLS = 10                  # <-- how many scroll steps
SCROLL_PAUSE_SEC = 4            # <-- wait after each scroll
STOP_AFTER_NO_NEW = 5             # <-- stop after N scrolls with no new posts

OUTFILE = f"threads_{TARGET_USERNAME}_raw.json"

def find_connection_with_page_info(obj):
    """Find a Relay connection dict that has both 'edges' and 'page_info'."""
    if isinstance(obj, dict):
        if "edges" in obj and "page_info" in obj:
            return obj
        for v in obj.values():
            found = find_connection_with_page_info(v)
            if found:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = find_connection_with_page_info(it)
            if found:
                return found
    return None

async def scrape_by_scrolling(username: str):
    collected_edges = []
    seen_post_keys = set()   # avoid duplicates
    no_new_counter = 0

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(storage_state="threads_state.json")
        page = await context.new_page()

        # Capture graphql responses
        async def on_response(resp):
            nonlocal collected_edges, seen_post_keys, no_new_counter
            try:
                if "/graphql/query" not in resp.url:
                    return
                if resp.status != 200:
                    return
                ctype = (resp.headers.get("content-type") or "").lower()
                if "application/json" not in ctype:
                    # sometimes still json; we'll try anyway
                    pass

                data = await resp.json()
                conn = find_connection_with_page_info(data)
                if not conn:
                    return

                edges = conn.get("edges") or []
                new_count = 0

                for e in edges:
                    # Create a stable-ish key; keep raw edge regardless
                    node = (e or {}).get("node") or {}
                    key = node.get("id") or node.get("pk") or json.dumps(node, sort_keys=True)[:200]
                    if key not in seen_post_keys:
                        seen_post_keys.add(key)
                        collected_edges.append(e)
                        new_count += 1

                if new_count > 0:
                    no_new_counter = 0  # reset when we get new content

            except Exception:
                # swallow parsing errors silently; Threads responses vary
                return

        page.on("response", on_response)

        # Go to profile
        url = f"https://www.threads.com/@{username}?hl=en"
        await page.goto(url, wait_until="networkidle")

        # Scroll loop
        for i in range(MAX_SCROLLS):
            # Scroll down
            await page.mouse.wheel(0, 2200)
            await asyncio.sleep(SCROLL_PAUSE_SEC)

            # If no new posts for a while, stop
            no_new_counter += 1
            if no_new_counter >= STOP_AFTER_NO_NEW:
                break

        await browser.close()

    # Save raw edges
    with open(OUTFILE, "w", encoding="utf-8") as f:
        json.dump(collected_edges, f, ensure_ascii=False, indent=2)

    return OUTFILE, len(collected_edges)

outfile, n = await scrape_by_scrolling(TARGET_USERNAME)
print("Saved:", outfile)
print("Edges collected:", n)


Saved: threads_nike_raw.json
Edges collected: 11


In [ ]:
import asyncio
import json
from playwright.async_api import async_playwright

In [5]:

# Configuration
TARGET_USERNAME = "nike"
TARGET_USER_ID = "63082097539"  # Add your user ID here
MAX_PAGES = 5  # Number of pages to scrape
OUTPUT_DIR = "threads_data"
DELAY_BETWEEN_PAGES = 2  # seconds


async def scrape_threads(username, user_id):
    """Main scraper function."""
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    all_pages_data = []
    
    async with async_playwright() as p:
        # Launch browser with saved auth state
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="threads_state.json")
        
        print(f"✅ Using user ID: {user_id}")
        
        # Scrape pages
        cursor = None
        page_num = 1
        
        while page_num <= MAX_PAGES:
            print(f"\n📄 Scraping page {page_num}...")
            
            # Create new page for each request
            page = await context.new_page()
            
            # Capture the response
            response_data = None
            
            async def handle_route(route, request):
                # Continue the request normally
                response = await route.fetch()
                
                # Try to capture response body
                try:
                    if "/graphql/query" in request.url:
                        post_data = request.post_data
                        
                        if post_data and "BarcelonaProfileThreadsTabRefetchable" in post_data:
                            body = await response.body()
                            nonlocal response_data
                            response_data = json.loads(body.decode('utf-8'))
                            print(f"✅ Captured response for page {page_num}")
                except Exception as e:
                    print(f"⚠️ Error in route handler: {e}")
                
                # Fulfill with the original response
                await route.fulfill(response=response)
            
            # Intercept requests
            await page.route("**/graphql/query", handle_route)
            
            # Navigate to trigger request
            if cursor:
                # For pagination, go to profile and scroll
                await page.goto(f"https://www.threads.com/@{username}")
                await page.wait_for_load_state("networkidle")
                
                # Scroll down to trigger next page load
                for i in range(5):
                    await page.mouse.wheel(0, 3000)
                    await asyncio.sleep(0.5)
                    if response_data:
                        break
            else:
                # First page
                await page.goto(f"https://www.threads.com/@{username}")
                await page.wait_for_load_state("networkidle")
            
            await asyncio.sleep(DELAY_BETWEEN_PAGES)
            
            if not response_data:
                print("❌ No data captured, stopping")
                await page.close()
                break
            
            # Save page data
            page_file = f"{OUTPUT_DIR}/{username}_page_{page_num}.json"
            with open(page_file, "w", encoding="utf-8") as f:
                json.dump(response_data, f, ensure_ascii=False, indent=2)
            print(f"💾 Saved to {page_file}")
            
            all_pages_data.append(response_data)
            
            # Extract next cursor
            try:
                page_info = response_data["data"]["mediaData"]["page_info"]
                has_next = page_info.get("has_next_page", False)
                cursor = page_info.get("end_cursor") or page_info.get("cursor", "")
                
                print(f"📊 Has next page: {has_next}")
                print(f"🔗 Next cursor: {cursor[:50]}..." if cursor else "🔗 No cursor")
                
                if not has_next or not cursor:
                    print("✅ Reached end of posts")
                    await page.close()
                    break
            
            except Exception as e:
                print(f"❌ Error extracting pagination: {e}")
                await page.close()
                break
            
            await page.close()
            page_num += 1
        
        await browser.close()
    
    # Save combined data
    combined_file = f"{OUTPUT_DIR}/{username}_all_pages.json"
    with open(combined_file, "w", encoding="utf-8") as f:
        json.dump(all_pages_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Scraping complete!")
    print(f"📁 Individual pages: {OUTPUT_DIR}/{username}_page_*.json")
    print(f"📁 Combined file: {combined_file}")
    print(f"📊 Total pages scraped: {len(all_pages_data)}")
    
    return all_pages_data


# Run the scraper (use this in Jupyter)
result = await scrape_threads(TARGET_USERNAME, TARGET_USER_ID)

✅ Using user ID: 63082097539

📄 Scraping page 1...
✅ Captured response for page 1
💾 Saved to threads_data/nike_page_1.json
📊 Has next page: True
🔗 Next cursor: QVFDS19TWWg2b1VCNXc2MFEtNUM3Vkdvd1JZSm9ueUlOd0tRS2...

📄 Scraping page 2...
✅ Captured response for page 2
💾 Saved to threads_data/nike_page_2.json
📊 Has next page: True
🔗 Next cursor: QVFCdm5CX1E1WHBMQWIwS2VwaVZya3NmR2hIVzRuWGFkTEtGVm...

📄 Scraping page 3...
✅ Captured response for page 3
💾 Saved to threads_data/nike_page_3.json
📊 Has next page: True
🔗 Next cursor: QVFCbFJMa0NTMWlHOEFRQzRqeVpKNWI4S0twN25GZzNVWEt3c0...

📄 Scraping page 4...
✅ Captured response for page 4
💾 Saved to threads_data/nike_page_4.json
📊 Has next page: True
🔗 Next cursor: QVFCZDZkZUcxbWJGZmdqeFgyMHE2M2pwd2VSd250cEpTcU1vVF...

📄 Scraping page 5...
✅ Captured response for page 5
💾 Saved to threads_data/nike_page_5.json
📊 Has next page: True
🔗 Next cursor: QVFEd2s3T2poRzF0Y2M4aXE1bER6TXN2ZWh5OWc0bl9HV1FfNW...

✅ Scraping complete!
📁 Individual pages: thre

In [8]:
import asyncio
import json
from playwright.async_api import async_playwright
from urllib.parse import parse_qs, urlencode

# Configuration
TARGET_USERNAME = "nike"
TARGET_USER_ID = "63082097539"  # Add your user ID here
MAX_PAGES = 20  # Number of pages to scrape
OUTPUT_DIR = "threads_data"
DELAY_BETWEEN_PAGES = 3  # seconds


async def scrape_threads(username, user_id):
    """Main scraper function."""
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    all_pages_data = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="threads_state.json")
        
        print(f"✅ Using user ID: {user_id}")
        
        cursor = None
        page_num = 1
        
        while page_num <= MAX_PAGES:
            print(f"\n📄 Scraping page {page_num}...")
            
            page = await context.new_page()
            response_data = None
            request_count = 0
            
            async def handle_route(route, request):
                nonlocal response_data, request_count, cursor
                
                # Check if this is the GraphQL request we want
                if "/graphql/query" not in request.url:
                    await route.continue_()
                    return
                
                post_data = request.post_data
                if not post_data or "BarcelonaProfileThreadsTabRefetchable" not in post_data:
                    await route.continue_()
                    return
                
                # Parse the form data
                parsed_data = parse_qs(post_data)
                
                # If we have a cursor and this is not the first page, modify the request
                if cursor and page_num > 1:
                    # Parse existing variables
                    variables_str = parsed_data.get('variables', ['{}'])[0]
                    variables = json.loads(variables_str)
                    
                    # Update the cursor
                    variables['after'] = cursor
                    
                    # Rebuild the request body
                    new_data = {}
                    for key, value in parsed_data.items():
                        if key == 'variables':
                            new_data[key] = json.dumps(variables)
                        else:
                            new_data[key] = value[0] if isinstance(value, list) else value
                    
                    new_post_data = urlencode(new_data)
                    
                    print(f"🔄 Modified request with cursor: {cursor[:30]}...")
                    
                    # Continue with modified request
                    await route.continue_(post_data=new_post_data)
                else:
                    # Continue normally
                    await route.continue_()
                
                request_count += 1
            
            async def handle_response(response):
                nonlocal response_data
                
                try:
                    if "/graphql/query" not in response.url:
                        return
                    
                    request = response.request
                    post_data = request.post_data
                    
                    if post_data and "BarcelonaProfileThreadsTabRefetchable" in post_data:
                        # Get response body
                        body = await response.body()
                        data = json.loads(body.decode('utf-8'))
                        
                        # Only capture if we haven't captured yet
                        if not response_data:
                            response_data = data
                            print(f"✅ Captured response for page {page_num}")
                
                except Exception as e:
                    print(f"⚠️ Error capturing response: {e}")
            
            # Set up interception
            await page.route("**/graphql/query", handle_route)
            page.on("response", handle_response)
            
            # Navigate to profile
            await page.goto(f"https://www.threads.com/@{username}")
            await page.wait_for_load_state("networkidle")
            
            # Scroll a bit to ensure content loads
            await page.mouse.wheel(0, 1000)
            await asyncio.sleep(DELAY_BETWEEN_PAGES)
            
            await page.close()
            
            if not response_data:
                print("❌ No data captured, stopping")
                break
            
            # Save page data
            page_file = f"{OUTPUT_DIR}/{username}_page_{page_num}.json"
            with open(page_file, "w", encoding="utf-8") as f:
                json.dump(response_data, f, ensure_ascii=False, indent=2)
            print(f"💾 Saved to {page_file}")
            
            all_pages_data.append(response_data)
            
            # Extract next cursor
            try:
                page_info = response_data["data"]["mediaData"]["page_info"]
                has_next = page_info.get("has_next_page", False)
                new_cursor = page_info.get("end_cursor") or page_info.get("cursor", "")
                
                print(f"📊 Has next page: {has_next}")
                print(f"🔗 Next cursor: {new_cursor[:50]}..." if new_cursor else "🔗 No cursor")
                
                if not has_next or not new_cursor:
                    print("✅ Reached end of posts")
                    break
                
                # Update cursor for next iteration
                cursor = new_cursor
            
            except Exception as e:
                print(f"❌ Error extracting pagination: {e}")
                break
            
            page_num += 1
        
        await browser.close()
    
    # Save combined data
    combined_file = f"{OUTPUT_DIR}/{username}_all_pages.json"
    with open(combined_file, "w", encoding="utf-8") as f:
        json.dump(all_pages_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Scraping complete!")
    print(f"📁 Individual pages: {OUTPUT_DIR}/{username}_page_*.json")
    print(f"📁 Combined file: {combined_file}")
    print(f"📊 Total pages scraped: {len(all_pages_data)}")
    
    return all_pages_data


# Run the scraper (use this in Jupyter)
Result = await scrape_threads(TARGET_USERNAME, TARGET_USER_ID)

✅ Using user ID: 63082097539

📄 Scraping page 1...
✅ Captured response for page 1
💾 Saved to threads_data/nike_page_1.json
📊 Has next page: True
🔗 Next cursor: QVFESDJaX1ZmdGNoUXdGNEp4LWNhcWJCRXFmU1FKeG42SWNOaU...

📄 Scraping page 2...
🔄 Modified request with cursor: QVFESDJaX1ZmdGNoUXdGNEp4LWNhcW...
✅ Captured response for page 2
💾 Saved to threads_data/nike_page_2.json
📊 Has next page: True
🔗 Next cursor: QVFEQVVKdXA2UmtCN3pYRTJUT2E1M0dvbEJVemdsVHpXUjYycm...

📄 Scraping page 3...
🔄 Modified request with cursor: QVFEQVVKdXA2UmtCN3pYRTJUT2E1M0...
✅ Captured response for page 3
💾 Saved to threads_data/nike_page_3.json
📊 Has next page: True
🔗 Next cursor: QVFDUVg3Q0ZsRjQ2aXBpcmQ3TnNwUXlxYUgxZDI4dFozMGl6NV...

📄 Scraping page 4...
🔄 Modified request with cursor: QVFDUVg3Q0ZsRjQ2aXBpcmQ3TnNwUX...
✅ Captured response for page 4
💾 Saved to threads_data/nike_page_4.json
📊 Has next page: True
🔗 Next cursor: QVFBLWk3UWZ5Mmo5Z3Y0YjBEZ3laakNFZ3puX2lBS09IU3ltLV...

📄 Scraping page 5...
🔄 Modified 